In [1]:
import requests
from bs4 import BeautifulSoup
from typing import List, Dict
import re


# -----------------------------
# 1. Fetch HTML
# -----------------------------
def fetch_page(url: str) -> str:
    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; TripReportBot/1.0)"
    }
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    return response.text


# -----------------------------
# 2. Parse posts from page
# -----------------------------
def parse_posts(html: str) -> List[Dict]:
    soup = BeautifulSoup(html, "html.parser")

    posts = []

    # High Sierra Topix uses "postbody" class for content
    post_bodies = soup.find_all("div", class_="postbody")

    for post in post_bodies:
        content_div = post.find("div", class_="content")

        if not content_div:
            continue

        text = content_div.get_text(separator="\n", strip=True)

        posts.append({
            "text": text
        })

    return posts


# -----------------------------
# 3. Filter for Wind River mentions
# -----------------------------
def filter_wind_river(posts: List[Dict]) -> List[Dict]:
    pattern = re.compile(r"\bwind rivers?\b", re.IGNORECASE)

    filtered = []
    for post in posts:
        if pattern.search(post["text"]):
            filtered.append(post)

    return filtered


# -----------------------------
# 4. Main pipeline
# -----------------------------
def scrape_trip_report(url: str) -> List[Dict]:
    html = fetch_page(url)
    posts = parse_posts(html)
    wind_posts = filter_wind_river(posts)

    return wind_posts

In [2]:
url = "https://www.highsierratopix.com/community/viewtopic.php?t=21021"

results = scrape_trip_report(url)


In [3]:
info = []

for i, post in enumerate(results, 1):
    print(f"\n--- Post {i} ---\n")
    print(post["text"][:500])  # preview
    info.append({"source_url": url,
    "text": post["text"],
    "region_hint": "wind rivers"})


--- Post 1 ---

2020 WR(4)  Bear Basin (cut short) and Big Sandy Loop
8/13 – 8/21
Back in Lander after my third trip, I felt great!  My feet were back to normal and I was looking forward to getting to higher altitudes, off -trail and into the more difficult northern part of the Wind River Range.   In addition I would have companions on these last two trips.  Being one day late off of the 3rd trip, I only had one day in town.
My next trip had two options; staying in Bear Basin for more fishing, or quickly throug

--- Post 2 ---

A footnote.
Just after Labor Day a severe storm in the Wind Rivers did a lot of damage.  The Big Sandy Trail, along with Scab Creek Trail, trail from Elkhart Park and Green River Lakes trail now have over 200 downed trees and there were several rescues conducted.  It is hard to say what other trails have been impacted. There are so many standing dead trees in the +/- 9000-foot elevation band that I fear many more trails were severely damaged.
My friend Steve, w

In [11]:
from openai import OpenAI
import json
import os
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")

client = OpenAI(api_key=OPENAI_API_KEY)

def parse_trip_report(text: str) -> dict:
    prompt = f"""
Extract structured fish observations from this trip report. There may be multiple fish observations.
Return a single observation for every lake-species-date pair. Only lake_name and species are required.

Return ONLY valid JSON with a list of objects of the following form:
- lake_name
- species
- count (int/null)
- length (int/null)
- date
- notes

Trip report:
{text}
"""

    response = client.responses.create(
        model="gpt-4o-mini",
        input=prompt,
    )

    output_text = response.output[0].content[0].text
    return output_text
    # try:
    #     print(output_text)
    #     return json.loads(output_text)
    # except json.JSONDecodeError:
    #     print("Failed to parse JSON")
    #     return None

response = parse_trip_report(info[0]["text"])
stripped = response[response.find("["): response.rfind("]")+1]
observation_list = json.loads(stripped)

In [24]:
stripped = response[response.find("["): response.rfind("]")+1]
observation_list = json.loads(stripped)
valid_list = []
for obs in observation_list:
    if (obs['species'] is not None and obs['species'] != ''):
        valid_list.append(obs)
valid_list

[{'lake_name': 'Upper Crescent Lake',
  'species': 'cutthroat',
  'count': 3,
  'length': 12,
  'date': '2020-08-13',
  'notes': None},
 {'lake_name': 'Native Lake',
  'species': 'golden',
  'count': None,
  'length': None,
  'date': None,
  'notes': 'hard to catch'},
 {'lake_name': 'Bear Lake',
  'species': 'rainbow',
  'count': None,
  'length': None,
  'date': None,
  'notes': 'supposedly big'}]